# 🧠 Workshop: Building Blocks for AI Agents

## NLP Pipeline + Probabilistic Language Models (90-Minute Team Lab)

**Objective:**
Work in teams of 3 to build a small NLP pipeline and implement unigram and bigram models, culminating in estimating sentence probabilities. Submit your completed Jupyter Notebook via a GitHub link (with `.git` at the end).

## Part 1 – NLP Pipeline

### Step 1: Select and Load a Corpus

Select a corpus from `nltk`, or upload your own text documents. Ensure your vocabulary size exceeds 2000 words.

In [1]:
import os
#import nltk
#from nltk.corpus import reuters

file_name = "Harry Potter and the Sorcerer's Stone.txt"

try:
    # read the file content
    with open(file_name, "r", encoding="utf-8") as f:
        corpus_text = f.read()
    
    # Checkover 2000 words later
    print(f"Success! The file '{file_name}' is loaded.")
    print(f"Total characters in file: {len(corpus_text)}")

except FileNotFoundError:
    # Error message
    print(f"Error: The file '{file_name}' was not found. Please check the file name.")



Success! The file 'Harry Potter and the Sorcerer's Stone.txt' is loaded.
Total characters in file: 442744


**👨‍🏫 Professor Talking Point:** This corpus is pre-tokenized and covers multiple topics. It’s a good fit to get us above the 2,000-word vocabulary requirement.

### Step 2: Collect and Preprocess Documents

Convert your corpus into tokens and compute the vocabulary size.

In [2]:
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Set the folder path for downloading NLTK data
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')

# Download tools for splitting sentences and words
print("Downloading 'punkt' tokenizer...")
nltk.download('punkt', download_dir=nltk_data_path, force=True)
nltk.download('punkt_tab', download_dir=nltk_data_path, force=True)

# tools path
if nltk_data_path not in nltk.data.path:
    nltk.data.path.append(nltk_data_path)

# letters to lowercase and split the text into words
tokens = word_tokenize(corpus_text.lower())

# Create a set --unique words
# 'vocab' counts different words
vocab = set(tokens)

# Show 
print(f"Vocabulary size: {len(vocab)}")

[nltk_data] Downloading package punkt to c:\Users\DELL\Downloads\Proba
[nltk_data]     bilisticLanguageModels\ProbabilisticLanguageModels\nlt
[nltk_data]     k_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
c:\Users\DELL\Downloads\ProbabilisticLanguageModels\ProbabilisticLanguageModels\venv\Lib\site-packages\nltk\downloader.py:2395: RuntimeWarning: Security Warning [pathsec.ZipFile]: Path C:\Users\DELL\Downloads\ProbabilisticLanguageModels\ProbabilisticLanguageModels\nltk_data\tokenizers\punkt.zip allowed via CWD.
  zf = ZipFile(filename)
[nltk_data] Downloading package punkt_tab to c:\Users\DELL\Downloads\P
[nltk_data]     robabilisticLanguageModels\ProbabilisticLanguageModels
[nltk_data]     \nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
c:\Users\DELL\Downloads\ProbabilisticLanguageModels\ProbabilisticLanguageModels\venv\Lib\site-packages\nltk\downloader.py:2395: RuntimeWarning: Security Warning [pathsec.ZipFile]: Path C:\Users\DELL\Downloads\ProbabilisticLang

Vocabulary size: 6002


**👨‍🏫 Professor Talking Point:** Vocabulary size is important—it determines the richness of our model. Models trained on small vocabularies can't generalize well.

### Step 3: Implement Tokenizer

In [3]:
import re

# Define a function to split text into words
def simple_tokenizer(text):
    # 're.findall'--> finds all patterns in the text
    # '\b\w+\b' -->find whole words, ignore spaces and punctuation
    return re.findall(r'\b\w+\b', text.lower())

# Use new function
tokens = simple_tokenizer(corpus_text)

# Show result
print(f"First 10 tokens: {tokens[:10]}")

First 10 tokens: ['harry', 'potter', 'and', 'the', 'sorcerer', 's', 'stone', 'chapter', 'one', 'the']


**👨‍🏫 Professor Talking Point:** A simple regex tokenizer gives us control—this is useful when we need to understand every processing step.

### Step 4: Normalization, Stemming, and Stopword Removal

In [4]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string
import os

# Download common words (stopwords)
nltk_data_path = os.path.join(os.getcwd(), 'nltk_data')
nltk.download('stopwords', download_dir=nltk_data_path)

def normalize(tokens):
    # Get the list
    stop_words = set(stopwords.words('english'))
    
    # Create a tool to cut words to their roots
    stemmer = PorterStemmer()
    
    # Filter the words: remove stopwords and punctuation, then stem them
    return [stemmer.stem(word) for word in tokens if word not in stop_words and word not in string.punctuation]

# Clean tokens
normalized_tokens = normalize(tokens)

# Check the first 10 cleaned words
print(f"Before-normalization - First 10 tokens: {tokens[:10]}")
print(f"After-normalization - First 10 normalized tokens: {normalized_tokens[:10]}")

[nltk_data] Downloading package stopwords to c:\Users\DELL\Downloads\P
[nltk_data]     robabilisticLanguageModels\ProbabilisticLanguageModels
[nltk_data]     \nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Before-normalization - First 10 tokens: ['harry', 'potter', 'and', 'the', 'sorcerer', 's', 'stone', 'chapter', 'one', 'the']
After-normalization - First 10 normalized tokens: ['harri', 'potter', 'sorcer', 'stone', 'chapter', 'one', 'boy', 'live', 'mr', 'mr']


**👨‍🏫 Professor Talking Point:** Normalization makes the data more consistent and shrinks the vocabulary. This is essential for estimating reliable probabilities.

### Step 5: Inverted Index

An **Inverted Index** maps each unique word to all the positions it appears in the corpus. This is the backbone of how search engines work — instead of scanning every document, you look up the word directly.

In [9]:
from collections import defaultdict

# Build inverted index: each word maps to a list of positions it appears at
inverted_index = defaultdict(list)

for position, word in enumerate(normalized_tokens):
    inverted_index[word].append(position)

# Summary
print(f"Total unique words in index: {len(inverted_index)}")

# Test lookups with Harry Potter words
test_words = ["harri", "potter", "voldemort", "magic", "wand"]

for word in test_words:
    positions = inverted_index[word]
    print(f"'{word}' → appears {len(positions)} times | First 5 positions: {positions[:5]}")

Total unique words in index: 4022
'harri' → appears 1327 times | First 5 positions: [0, 490, 540, 547, 938]
'potter' → appears 115 times | First 5 positions: [1, 89, 91, 115, 120]
'voldemort' → appears 38 times | First 5 positions: [1458, 1479, 1497, 1503, 1580]
'magic' → appears 59 times | First 5 positions: [7995, 8106, 8127, 8427, 8569]
'wand' → appears 75 times | First 5 positions: [7947, 8144, 8979, 9076, 10425]


### Talking Point
The Inverted Index shows exactly where each word appears in the corpus by position.
For example, `'harri'` appears 132 times across the book — the index stores every
single one of those positions. This structure is what makes fast text search possible;
instead of reading the whole corpus, you jump directly to the positions you need.

## Part 2 – Probabilistic Language Models

### 📘 Unigram Model

A **Unigram Model** is a type of probabilistic language model that assumes each word in a sentence is **independent** of the words that came before it.

The probability of a sequence of words $w_1, w_2, ..., w_n$ is calculated as:

$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$

To estimate $P(w_i)$, we use the **Maximum Likelihood Estimate (MLE)**:

$$
P(w_i) = \frac{\text{count}(w_i)}{\sum_{j} \text{count}(w_j)}
$$

where $j$ is the total number of words in the corpus.

This is a strong simplification, but it provides a foundational baseline and helps reduce data sparsity in low-resource environments.

Here's how to implement it:


In [5]:
from collections import Counter
import pandas as pd

# Count frequency of normalized_tokens
unigram_counts = Counter(normalized_tokens)

# Count total Volume normalized_tokens
total_words = len(normalized_tokens)

#  Create probability of one word
def unigram_prob(word):
    # Probability = (Count of the word) / (Total words)
    return unigram_counts[word] / total_words


# Test with words from Harry Potter

# test_words = ["harri", "potter", "magic", "wand", "hagrid", "voldemort"]
test_words = ["harri", "potter", "magic", "wand", "hagrid", "voldemort", "Cedric"]

def generate_unigram_matrix(word_list):
    # Get probabilities
    data = []
    for word in word_list:
            # Get count from Counter
            count = unigram_counts[word]
            # Get probability
            prob = unigram_prob(word)
            
            data.append({
                "Word": word,
                "Count ": count,
                "Total ": total_words,
                "Formula": f"{count} / {total_words}",
                "Probability": f"{prob:.5f}"
            })
        
    # Create the table
    df = pd.DataFrame(data)
    return df

# show the result
unigram_matrix = generate_unigram_matrix(test_words)
unigram_matrix


,Word,Count,Total,Formula,Probability
0,harri,1327,40785,1327 / 40785,0.03254
1,potter,115,40785,115 / 40785,0.00282
2,magic,59,40785,59 / 40785,0.00145
3,wand,75,40785,75 / 40785,0.00184
4,hagrid,370,40785,370 / 40785,0.00907
5,voldemort,38,40785,38 / 40785,0.00093
6,Cedric,0,40785,0 / 40785,0.00000


### Talking Point
1.In Step 4, we used a tool called `PorterStemmer`.This tool cuts words to their "root" to make counting easier. It follows a rule: words ending in '-y' often change to '-i'.
2.I also tested the name "Cedric Diggory," a character who first appears in the 4th book. The probability table correctly shows a value of 0, which proves the model is accurate.

##### 📘 Why Are Unigram Probabilities So Low?

Unigram probabilities represent the **relative frequency** of individual words in the entire corpus:

$$
P(w_i) = \frac{\text{count}(w_i)}{\text{total number of tokens in the corpus}}
$$

In our case, the total number of tokens is quite large:

- **Total tokens:** 407,850  
- **Unique words (vocabulary size):** 6,002

Even if a word appears frequently, its probability will still be small relative to the total number of tokens.

For example:
- `"harri"` appears 132 times, with a probability of **0.03254**, or about **3.3%** of total tokens.  
- `"voldemort"` appears only 38 times, resulting in a smaller probability of **0.00093**.

These small values are expected when:
- The corpus is **large and diverse** (like Reuters).
- Many words appear **only once or twice**, which is common in natural language (known as Zipf's Law).

**Conclusion:**  
Low unigram probabilities do **not** indicate an error—they reflect a realistic distribution of word frequencies across a large corpus. This also highlights the need for smoothing when building more complex language models.


### 📘 Chain Rule with Unigrams

Using the **Chain Rule**, we estimate the probability of a sequence:
$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$
This is a simplifying assumption of complete independence (unrealistic but foundational).

**👨‍🏫 Professor Talking Point:** Unigram models assume word independence—useful but limited since word order is ignored.

In [6]:
def sentence_prob_unigram(sentence):
    # Split and clean the input sentence
    words = normalize(simple_tokenizer(sentence))
    #Start probability
    prob = 1.0
    for word in words:
        prob *= unigram_prob(word)
    return prob

test_sentence = "Hermione used the Sorcerer's Stone to defeat Voldemort."

# show the result
result = sentence_prob_unigram(test_sentence)
print(f"Sentence: '{test_sentence}'")
print(f"Tokens the model uses: {normalize(simple_tokenizer(test_sentence))}")
print(f"Unigram Probability: {result}")


Sentence: 'Hermione used the Sorcerer's Stone to defeat Voldemort.'
Tokens the model uses: ['hermion', 'use', 'sorcer', 'stone', 'defeat', 'voldemort']
Unigram Probability: 4.8124142652816015e-19


### Talking Point
* I tested a "fake" plot:`Hermione used the Sorcerer's Stone to defeat Voldemort.The model calculated a probability of 4.8124142652816015e-19.This shows a key weakness of the Unigram model: it only cares if the words exist, not if the sentence makes sense or follows the actual story. As long as the words are in the book, the model will say the sentence is "possible," even if the order is wrong.

##### 📉 How the Unigram Model Works

The unigram model computes sentence probability as the **product of individual word probabilities**:

$$
P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i)
$$

Each word typically has a probability between 0.00001 and 0.01. When multiplying **10–20 small numbers together**, the final result becomes **exponentially smaller**, approaching zero for longer sentences.

##### 🧪 Impact of Preprocessing (Step 4)

The normalization step involves:

* Lowercasing
* **Stop word removal** (e.g., "the", "of", "for", "said")
* **Stemming** (e.g., "management" → "manag")
* **Punctuation removal**

This reduces the number of words used in the calculation. While this makes the vocabulary smaller and more manageable, it also means:

* **Common but removed words** (like "the") don’t contribute to the probability.
* **Stemmed forms** may not match original unigrams perfectly (e.g., “sino-chilean” becomes `sinochilean` or `sino` and `chilean`, depending on the tokenizer).

So even though the sentence appears long, **only 7–12 stemmed and filtered tokens** may remain after preprocessing—yet each one still has a very small individual probability.

##### ✅ Key Takeaways

* Low sentence probabilities are **normal** in unigram models, especially for longer sentences.
* The **multiplicative nature** of probability and the **sparsity of natural language** lead to very small final values.
* These limitations are one reason why more advanced models (like bigrams or neural LMs) are needed for realistic NLP applications.


### 📘 Bigram Model with MLE – Mathematical Explanation

The **Bigram Model** assumes the current word depends only on the previous word.
The MLE (Maximum Likelihood Estimate) for a bigram $(w_{i-1}, w_i)$ is:
$$
P(w_i | w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}
$$

**👨‍🏫 Professor Talking Point:** This simple multiplication illustrates the chain rule, but we’ll soon see how to improve this with context.

### 📘 Sentence Probability with Bigram Model – Mathematical Explanation

Using the bigram model and chain rule:
$$
P(w_1, w_2, ..., w_n) = P(w_1) \cdot P(w_2 | w_1) \cdot P(w_3 | w_2) \cdots P(w_n | w_{n-1})
$$
This models **local dependencies** between words.

In [7]:
from collections import defaultdict

bigram_counts = defaultdict(int)

for i in range(len(normalized_tokens) - 1):
    pair = (normalized_tokens[i], normalized_tokens[i + 1])
    # Records every "word pair"
    bigram_counts[pair] += 1
# word w2 appearing after w1.
def bigram_prob(w1, w2):
    return bigram_counts[(w1, w2)] / unigram_counts[w1] if unigram_counts[w1] > 0 else 0

**👨‍🏫 Professor Talking Point:** Bigram probabilities model word context, capturing more meaning than unigrams.

### Sentence Probability with Bigram Model

In [8]:
# score a sentence
def sentence_prob_bigram(sentence):
    words = normalize(simple_tokenizer(sentence))
    steps = []
    prob = 1.0
    for i in range(len(words) - 1):
            w1, w2 = words[i], words[i+1]
            
            pair_count = bigram_counts[(w1, w2)]
            w1_count = unigram_counts[w1]
            
            step_prob = bigram_prob(w1, w2)
            old_prob = prob
            prob *= step_prob
            
            steps.append({
                "Pair ": f"({w1}, {w2})",
                "Count(w1, w2)": pair_count,
                "Count(w1)": w1_count,
                "Prob P(w2|w1)": f"{step_prob:.5f}",
                "Calculation": f"{old_prob:.2e} * {step_prob:.5f}",
                "Running Total": f"{prob:.2e}"
            })
        
    return pd.DataFrame(steps)

# test_result = sentence_prob_bigram("Hermione Granger used the Sorcerer's Stone to defeat Voldemort.")
test_result = sentence_prob_bigram("Harry Potter has the Sorcerer's Stone .")

test_result

,Pair,"Count(w1, w2)",Count(w1),Prob P(w2|w1),Calculation,Running Total
0,"(harri, potter)",30,1327,0.02261,1.00e+00 * 0.02261,2.26e-02
1,"(potter, sorcer)",1,115,0.00870,2.26e-02 * 0.00870,1.97e-04
2,"(sorcer, stone)",14,17,0.82353,1.97e-04 * 0.82353,1.62e-04


### Talking Point
I compared two sentences,
* The first sentence about Hermione resulted in 0.00e+00. This happened because the word pair (use, sorcer) never appears in the book. In a Bigram model, if just one pair is missing, the total probability becomes zero.
* The second sentence about "Harry Potter" has a higher probability (1.62e-04). This is because pairs like (harri, potter) and (sorcer, stone) are very common in the text. This shows that Bigram models are very good at identifying "correct" phrases from the book, but they are very strict about word order.

**👨‍🏫 Professor Talking Point:** Estimating sentence probability using bigrams shows how sequence information improves prediction power.

## Part 3: The Workshop


One team member must push the final notebook to GitHub and send the `.git` URL to the instructor before the end of class.

## 🧠 Learning Objectives
- Implement the foundations of **Probabilistic Language Models** using real-world data during the NLP process.
- Build **Jupyter Notebooks** with well-structured code and clear Markdown documentation.
- Use **Git and GitHub** for collaborative version control and code sharing.
- Identify and articulate coding issues ("**talking points**") and insert them directly into peer notebooks.
- Practice **collaborative debugging**, professional peer feedback, and improve code quality.

## 🧩 Workshop Structure (90 Minutes)
1. **Instructor Use Case Introduction** *(20 min)* – Set up teams of 3 people. Read and understand the workshop, plus submission instructions. Seek assistance if needed.
2. **Team Jupyter Notebook Development** *(65 min)* – NLP Pipeline and four Probabilistic Language Model method implementations + Markdown documentation (work as teams)
3. **Push to GitHub** *(5 min)* – Teams commit and push the one notebook. **Make sure to include your names so it is easy to identify the team that developed the code**.
4. **Instructor Review** - The instructor will go around, take notes, and provide coaching as needed, during the **Peer Review Round**
5. **Email Delivery** *(1 min)* – Each team send the instructor an email **with the *.git link** to the GitHub repo **(one email/team)**. Subject on the email is: PROG8245 - Probabilistic Language Models Workshop, Team #_____.


## 💻 Submission Checklist
- ✅ `ProbabilisticLanguageModels.ipynb` with:
  - Demo code: Document Collection, Tokenizer, Normalization Pipeline, Inverted Index and the four methods.
  - Markdown explanations for each major step
  - **Labeled talking point(s)** (1-2 per concept)
- ✅ `README.md` with:
  - Dataset description
  - Team member names
  - Link to the dataset and license (if public)
- ✅ GitHub Repo:
  - Public repo named `ProbabilisticLanguageModels`
  - This is a group effort, so **choose one member of the team** to publish the repo
  - At least **one commit containing one meaningful talking point**

## 🧭 Conclusion

Today you’ve constructed your own basic language model. Next class, we’ll expand these ideas to explore **Large Language Models (LLMs)**—like ChatGPT—which learn patterns over **massive corpora** using **deep neural networks** instead of just counts.